# Dokumentasi Analisis *Value at Risk* (VaR) Portofolio Saham dengan Pendekatan ARMA–GARCH

**Tim STAVANTS — Data Analytics Competition (DAC)**

---

## Ringkasan

Notebook ini mendokumentasikan secara runtut seluruh tahapan analisis risiko pasar (*market risk*) pada portofolio saham menggunakan kerangka **Value at Risk (VaR)** berbasis model **ARMA–GARCH**. Estimasi VaR dilakukan baik pada level saham individual maupun pada level portofolio, dengan dua metode portofolio yang dibandingkan: pendekatan **Varians-Kovarians (parametrik)** dan **simulasi Monte Carlo**.

## Tujuan Analisis

1. Mempersiapkan dan membersihkan data harga penutupan (*closing price*) historis sejumlah saham, termasuk penanganan nilai hilang (*missing value*) secara sistematis.
2. Menguji asumsi statistik dasar pemodelan deret waktu finansial (stasioneritas, efek ARCH/heteroskedastisitas bersyarat).
3. Menentukan ordo model ARMA terbaik untuk setiap saham berdasarkan kriteria informasi (AIC).
4. Membangun model gabungan **ARMA (persamaan rataan) – GARCH (persamaan varians)** untuk menangkap dinamika volatilitas *log-return*.
5. Mengestimasi **VaR 95% dan VaR 99%** untuk horizon satu hari ke depan, baik per saham maupun portofolio.
6. Mengevaluasi efek diversifikasi portofolio dengan membandingkan risiko individual terhadap risiko gabungan.

## Struktur Notebook

| Bagian | Deskripsi |
|---|---|
| 1. Pemuatan Data | Pengambilan dataset mentah |
| 2. Persiapan Data | Pembersihan, konsistensi tipe data, dan penyesuaian rentang waktu |
| 3. Eksplorasi & Deteksi Keanehan | Visualisasi awal pola harga saham |
| 4. Penanganan Missing Value | Deteksi dan imputasi nilai hilang pada *log-return* |
| 5. Validasi Hasil Imputasi | Pembuktian bahwa imputasi menjaga karakteristik statistik data asli |
| 6. Uji Asumsi Statistik | Stasioneritas (ADF) dan efek ARCH (heteroskedastisitas) |
| 7. Pemilihan Ordo ARMA | Pencarian model rataan terbaik per saham |
| 8. Pemodelan ARMA–GARCH & Estimasi VaR | Pipeline *sequential* dan *simultaneous*, beserta uji diagnostik |
| 9. Pemilihan Model Hibrida | Penggabungan hasil dua pipeline berdasarkan kelolosan uji diagnostik dan AIC |
| 10. Simulasi Kerugian Nominal | Konversi VaR persentase ke estimasi kerugian Rupiah |
| 11. Analisis Risiko Portofolio | Korelasi antar-saham, VaR Varians-Kovarians, dan VaR Monte Carlo |
| 12. Komparasi Risiko | Evaluasi efek diversifikasi portofolio |

---


## 1. Pemuatan Data

Dataset yang digunakan berupa data historis harga saham harian (format CSV) yang diunduh dari penyimpanan *cloud* (Google Drive) menggunakan utilitas `gdown`. Data kemudian dibaca ke dalam `pandas.DataFrame` dengan pemisah kolom (`;`) yang sesuai dengan format berkas sumber.

In [ ]:
!pip install gdown
!gdown --id 1iEuypp19Q9mBLLERR9E6Sizbk-YiDWv9

In [ ]:
import pandas as pd
df = pd.read_csv("/content/Dataset DAC .csv", sep=';')

## 2. Persiapan Data (*Data Preparation*)

Sebelum dianalisis lebih lanjut, data mentah perlu disesuaikan agar konsisten dan bebas dari kesalahan format. Tahapan yang dilakukan meliputi:

- **Konsistensi tipe data**: nama saham dirapikan dari spasi berlebih, kolom tanggal dikonversi ke format `datetime`, dan data diurutkan berdasarkan nama saham dan tanggal.
- **Konsistensi nama kolom**: menghapus spasi berlebih pada nama kolom.

In [ ]:
# Konsistensi Tipe Data
df['Stock_Name'] = df['Stock_Name'].str.strip()
df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y %H:%M')
df = df.sort_values(['Stock_Name','Date']).reset_index(drop=True)

In [ ]:
df.columns = df.columns.str.strip()

### 2.1 Pengecekan Rentang Tanggal Tiap Saham

Karena setiap saham berpotensi memiliki tanggal pencatatan (*listing*) yang berbeda, perlu dilakukan pengecekan rentang tanggal awal dan akhir data yang tersedia untuk masing-masing saham. Data terlebih dahulu dipivot agar setiap kolom merepresentasikan satu saham (`Stock_Name`) dengan nilai harga penutupan (`Close`).

In [ ]:
print('=== RANGE TANGGAL TIAP SAHAM ===')

price = df.pivot(index='Date', columns='Stock_Name', values='Close')
price.columns.name = None

for stock in price.columns:
    s = price[stock]

    first_date = s.first_valid_index()
    last_date  = s.last_valid_index()

    first_str = first_date.date() if first_date else 'N/A'
    last_str  = last_date.date() if last_date else 'N/A'

    print(f'{stock:<6} | start: {first_str} | end: {last_str}')

### 2.2 Penyesuaian Rentang Waktu Data

Untuk menjaga keaslian dan keterbandingan antar-saham, dilakukan pemotongan data agar seluruh saham memiliki titik awal pengamatan yang sama, yakni mulai **1 Januari 2014**. Hal ini mengantisipasi saham yang baru tercatat (*listing*) pada periode yang lebih belakangan dibandingkan saham lainnya.

In [ ]:
# pastikan index datetime
price.index = pd.to_datetime(price.index)

# cut dari 2014
price_cut = price.loc['2014-01-01':]

print("Mulai dari:", price_cut.index.min().date())
print("\nShape sebelum:", price.shape)
print("Shape sesudah:", price_cut.shape)

## 3. Eksplorasi dan Deteksi Keanehan Data

Sebagai langkah pemeriksaan awal (*sanity check*), pola pergerakan harga salah satu saham divisualisasikan untuk mengidentifikasi adanya anomali, seperti lonjakan harga ekstrem atau pola yang tidak wajar akibat kesalahan input data.

In [ ]:
import matplotlib.pyplot as plt

# pilih 1 saham, misal BBCA
col = 'MAPI'

plt.figure(figsize=(12,5))
plt.plot(price_cut.index, price_cut[col])

plt.title(f'Price - {col}')
plt.xlabel('Date')
plt.ylabel('Price')
plt.grid()

plt.show()

## 4. Pemeriksaan Nilai Hilang (*Missing Value*)

### 4.1 Eksplorasi Visual Awal

Sebelum melakukan deteksi nilai hilang secara sistematis, dilakukan visualisasi pola pergerakan harga salah satu saham untuk mendapatkan gambaran awal kondisi data.

In [ ]:
import matplotlib.pyplot as plt

# pilih 1 saham
stock = 'ANTM'

plt.figure(figsize=(15, 5))

plt.plot(price_cut.index, price_cut[stock])

plt.xlabel('Date')
plt.ylabel('Close Price')
plt.title(f'Stock Price Over Time ({stock})')
plt.show()

### 4.2 Resampling Harian untuk Mendeteksi Tanggal yang Hilang

Data historis saham umumnya hanya tercatat pada hari perdagangan (*trading days*), sehingga untuk mendeteksi adanya tanggal yang hilang secara eksplisit, data perlu di-*resample* ke frekuensi harian penuh (*calendar days*). Proses ini juga membuang periode sebelum saham tersebut resmi tercatat (*listing*), agar tidak dianggap sebagai data yang hilang.

In [ ]:
# pilih saham dari pivot
stock = 'ANTM'

# ambil series
s = price_cut[stock].copy()

# buang bagian sebelum listing (penting!)
s = s.loc[s.first_valid_index():]

# buat full date range harian
full_date_range = pd.date_range(start=s.index.min(),
                               end=s.index.max(),
                               freq='D')

# reindex → munculin missing
df_daily = s.reindex(full_date_range)

### 4.3 Visualisasi Tanggal yang Hilang

Tanggal-tanggal dengan nilai hilang ditandai dengan garis vertikal pada grafik harga, sehingga sebaran dan kepadatan *missing value* dapat diamati secara visual.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

stock = 'MAPI'

# handle kalau df_daily bentuknya DataFrame atau Series
if isinstance(df_daily, pd.DataFrame):
    y = df_daily[stock] if stock in df_daily.columns else df_daily.iloc[:, 0]
else:
    y = df_daily

plt.figure(figsize=(15, 5))

# plot utama
plt.plot(y.index, y, label='Close Price', color='blue')

# ambil missing
missing_mask = y.isna()
missing_dates = y.index[missing_mask]

# garis vertikal (lebih efisien dari loop)
plt.vlines(missing_dates,
           ymin=y.min(),
           ymax=y.max(),
           colors='red',
           alpha=0.1,
           linewidth=0.5)

# full range
plt.xlim(y.index.min(), y.index.max())

# legend custom
missing_legend = mlines.Line2D([], [], color='red', alpha=0.3, linewidth=2, label='M issing Date')
price_legend = mlines.Line2D([], [], color='blue', label='Close Price')

plt.xlabel('Date')
plt.ylabel('Close Price')
plt.title(f'Missing Dates Highlighted ({stock})')
plt.legend(handles=[price_legend, missing_legend])

plt.tight_layout()
plt.show()

## 5. Transformasi Log-Return

Sebelum dilakukan imputasi, harga saham terlebih dahulu ditransformasikan ke dalam bentuk **log-return**:

$$r_t = \ln\left(\frac{P_t}{P_{t-1}}\right)$$

Transformasi ini dilakukan karena beberapa alasan metodologis:

1. **Stasioneritas**: log-return cenderung lebih stasioner dibandingkan harga mentah, yang merupakan prasyarat penting bagi pemodelan ARMA–GARCH.
2. **Stabilisasi varians**: transformasi logaritmik membantu menstabilkan skala data, terutama pada saham dengan rentang harga yang berbeda jauh.
3. **Imputasi yang lebih andal**: nilai hilang diimputasi pada skala log-return, bukan pada skala harga, untuk menjamin hasil rekonstruksi yang lebih konsisten secara statistik.

In [ ]:
import numpy as np
# log return
log_return = np.log(price_cut / price_cut.shift(1))

Selanjutnya, jumlah nilai hilang pada setiap saham diperiksa untuk mengetahui besaran masalah yang harus ditangani pada tahap imputasi.

In [ ]:
log_return.isna().sum().sort_values(ascending=False)

## 6. Pipeline Imputasi Data Hilang

Untuk menangani nilai hilang pada *log-return*, digunakan pipeline imputasi multi-tahap yang dirancang untuk menjaga karakteristik statistik data asli (rata-rata, volatilitas, dan pola musiman), bukan sekadar mengisi nilai kosong secara naif. Pipeline ini terdiri atas lima tahap berurutan:

1. **Rolling Smoothing** — pengisian awal menggunakan rata-rata bergerak (*rolling mean*) dengan jendela waktu bertingkat (7, 30, 90, dan 180 hari) untuk menangkap pola lokal dengan berbagai skala waktu.
2. **Volatility-Aware Fill** — pengisian sisa nilai hilang menggunakan rata-rata dan deviasi standar bergerak, ditambah komponen acak (*random noise* berdistribusi normal) agar volatilitas data tidak hilang akibat imputasi yang terlalu "halus".
3. **Seasonal Decomposition Fill** — pengisian berdasarkan dekomposisi musiman (*trend* + *seasonal*) menggunakan `seasonal_decompose`, untuk menangkap pola periodik (mingguan) dalam data.
4. **Polynomial Interpolation** — interpolasi polinomial orde-3 untuk merapikan sisa celah data yang belum tertangani oleh tahap sebelumnya.
5. **Final Clean-up** — pengisian akhir menggunakan *forward-fill* dan *backward-fill* sebagai jaring pengaman (*safety net*), memastikan tidak ada nilai hilang yang tersisa.

Selain fungsi inti imputasi, didefinisikan pula fungsi pendukung untuk merekonstruksi harga dari log-return yang telah diimputasi, serta fungsi bantu untuk membandingkan distribusi dan volatilitas data sebelum dan sesudah imputasi.

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.seasonal import seasonal_decompose

def rolling_stat_fill(series, window=20, z_random=True):
    s = series.copy()

    rolling_mean = s.rolling(window, min_periods=5).mean()
    rolling_std  = s.rolling(window, min_periods=5).std()

    nan_idx = s.isna()

    if z_random:
        z = np.random.normal(0, 1, size=len(s))
    else:
        z = 0

    s[nan_idx] = rolling_mean[nan_idx] + z[nan_idx] * rolling_std[nan_idx]

    return s

def seasonal_fill(series, period=5):
    s = series.copy()
    nan_idx = s.index[s.isna()]

    if len(nan_idx) == 0 or len(s.dropna()) < 2 * period:
        return s

    s_tmp = s.interpolate(method='linear', limit_direction='both')

    try:
        decomp = seasonal_decompose(
            s_tmp,
            model='additive',
            period=period,
            extrapolate_trend='freq'
        )

        baseline = decomp.trend.ffill().bfill() + decomp.seasonal
        s.loc[nan_idx] = baseline.loc[nan_idx]

    except:
        pass

    return s

def poly3_fill(series):
    try:
        return series.interpolate(method='polynomial', order=3, limit_direction='both')
    except:
        return series.interpolate(method='linear', limit_direction='both')

def impute_log_return(series):
    s = series.copy()

    # =========================
    # STEP 1: Rolling smoothing
    # =========================
    s = s.fillna(s.rolling(7,  min_periods=1, center=True).mean())
    s = s.fillna(s.rolling(30, min_periods=1, center=True).mean())
    s = s.fillna(s.rolling(90, min_periods=1, center=True).mean())
    s = s.fillna(s.rolling(180, min_periods=1, center=True).mean())

    # =========================
    # STEP 2: Volatility-aware
    # =========================
    s = rolling_stat_fill(s, window=20, z_random=True)

    # =========================
    # STEP 3: Seasonal pattern
    # =========================
    s = seasonal_fill(s, period=5)

    # =========================
    # STEP 4: Polynomial smoothing
    # =========================
    s = poly3_fill(s)

    # =========================
    # STEP 5: FINAL CLEAN (WAJIB)
    # =========================
    s = s.ffill().bfill()

    # safety net
    if s.isna().sum() > 0:
        s = s.interpolate(method='linear', limit_direction='both')

    return s

def impute_all_log_return(log_return_df):
    result = log_return_df.copy()
    report = []

    for col in log_return_df.columns:
        before = log_return_df[col].isna().sum()

        filled = impute_log_return(log_return_df[col])

        after = filled.isna().sum()

        result[col] = filled

        report.append({
            'stock': col,
            'missing_before': before,
            'missing_after': after
        })

    return result, pd.DataFrame(report)

def reconstruct_price(log_return_imputed, original_price):
    reconstructed = pd.DataFrame(index=log_return_imputed.index)

    for col in log_return_imputed.columns:
        r = log_return_imputed[col]

        # starting price
        p0 = original_price[col].dropna().iloc[0]

        prices = [p0]

        for i in range(1, len(r)):
            prices.append(prices[-1] * np.exp(r.iloc[i]))

        reconstructed[col] = prices

    return reconstructed

def check_missing(df):
    return df.isna().sum().sort_values(ascending=False)

import matplotlib.pyplot as plt
import seaborn as sns

def compare_distribution(original, imputed, col):
    sns.kdeplot(original[col].dropna(), label='Original')
    sns.kdeplot(imputed[col], label='Imputed')
    plt.title(col)
    plt.legend()
    plt.show()

def compare_volatility(original, imputed):
    return pd.DataFrame({
        'original_std': original.std(),
        'imputed_std': imputed.std()
    })

### 6.1 Eksekusi Pipeline Imputasi

Pipeline imputasi dijalankan pada seluruh kolom (saham), kemudian harga direkonstruksi kembali dari log-return yang telah diimputasi. Validasi dilakukan untuk memastikan tidak ada nilai hilang yang tersisa, dilanjutkan dengan pemeriksaan distribusi dan volatilitas sebagai indikator awal kualitas imputasi.

In [ ]:
# STEP 2: imputasi
log_return_imputed, report = impute_all_log_return(log_return)

print(report)

# STEP 3: reconstruct price
price_reconstructed = reconstruct_price(log_return_imputed, price_cut)

# STEP 4: validasi
print(check_missing(log_return_imputed))   # harus 0
print(check_missing(price_reconstructed))  # harus 0

# STEP 5: cek distribusi
compare_distribution(log_return, log_return_imputed, 'BBCA')

# STEP 6: cek volatility
print(compare_volatility(log_return, log_return_imputed))

## 7. Validasi Hasil Imputasi

Untuk memastikan bahwa proses imputasi tidak mendistorsi karakteristik statistik data, hasil imputasi divalidasi secara visual melalui dua fungsi pembanding:

- `plot_sebelum_imputasi`: menampilkan data asli beserta penanda tanggal yang hilang pada rentang waktu tertentu.
- `plot_sesudah_imputasi`: menampilkan data hasil imputasi pada rentang waktu yang sama, untuk diperbandingkan secara visual.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import pandas as pd

# ==========================================
# 1. PLOT SEBELUM IMPUTASI (Dengan Fitur Rentang Waktu)
# ==========================================
def plot_sebelum_imputasi(df_daily, stock='MAPI', start_date=None, end_date=None):
    if isinstance(df_daily, pd.DataFrame):
        y = df_daily[stock] if stock in df_daily.columns else df_daily.iloc[:, 0]
    else:
        y = df_daily

    # Fitur Zoom: Potong data berdasarkan rentang tanggal jika diinput
    if start_date is not None and end_date is not None:
        y = y.loc[start_date:end_date]

    plt.figure(figsize=(10, 5))

    # Plot utama
    plt.plot(y.index, y, label='Close Price', color='gray')

    # Ambil missing pada area yang sudah di-zoom
    missing_mask = y.isna()
    missing_dates = y.index[missing_mask]

    # Garis vertikal merah tipis
    if len(missing_dates) > 0:
        plt.vlines(missing_dates,
                   ymin=y.min(),
                   ymax=y.max(),
                   colors='red',
                   alpha=0.3,
                   linewidth=1)

    plt.xlim(y.index.min(), y.index.max())

    # Legend custom
    missing_legend = mlines.Line2D([], [], color='red', alpha=0.7, linewidth=2, label='Missing Date')
    price_legend = mlines.Line2D([], [], color='gray', label='Close Price')

    plt.xlabel('Date')
    plt.ylabel('Close Price')

    # Update Judul agar menampilkan rentang waktu jika ada
    title_text = f'Sebelum Imputasi: Missing Dates Highlighted ({stock})'
    if start_date and end_date:
        title_text += f'\n(Periode: {start_date} s/d {end_date})'

    plt.legend(handles=[price_legend, missing_legend])

    plt.tight_layout()
    plt.show()

# ==========================================
# 2. PLOT SESUDAH IMPUTASI (Dengan Fitur Rentang Waktu)
# ==========================================
def plot_sesudah_imputasi(df_imputed, stock='MAPI', start_date=None, end_date=None):
    if isinstance(df_imputed, pd.DataFrame):
        y = df_imputed[stock] if stock in df_imputed.columns else df_imputed.iloc[:, 0]
    else:
        y = df_imputed

    # Fitur Zoom: Potong data berdasarkan rentang tanggal jika diinput
    if start_date is not None and end_date is not None:
        y = y.loc[start_date:end_date]

    plt.figure(figsize=(10, 5))

    # Plot utama
    plt.plot(y.index, y, label='Imputed Close Price', color='blue')

    plt.xlim(y.index.min(), y.index.max())

    price_legend = mlines.Line2D([], [], color='blue', label='Imputed Close Price')

    plt.xlabel('Date')
    plt.ylabel('Close Price')

    # Update Judul agar menampilkan rentang waktu jika ada
    title_text = f'Sesudah Imputasi: Data Lengkap Siap Model ({stock})'
    if start_date and end_date:
        title_text += f'\n(Periode: {start_date} s/d {end_date})'

    plt.legend(handles=[price_legend])

    plt.tight_layout()
    plt.show()

Sebagai ilustrasi, validasi dilakukan pada saham **AMRT** untuk periode 2015–2018, yang memiliki celah data yang cukup signifikan.

In [ ]:
plot_sebelum_imputasi(log_return, stock='AMRT', start_date='2015-05-01', end_date='2018-01-01')

plot_sesudah_imputasi(log_return_imputed, stock='AMRT', start_date='2015-05-01', end_date='2018-01-01')

### 7.1 Pembuktian Visual Komprehensif

Sebagai pembuktian lebih menyeluruh, dilakukan tiga jenis visualisasi pembanding pada saham **AMRT**:

1. **Rekonstruksi trayektori log-return** — membandingkan data hasil imputasi dengan data observasi asli pada area yang sebelumnya memiliki nilai hilang, untuk menilai kecocokan pola volatilitas lokal.
2. **Preservasi struktur varians** — membandingkan volatilitas bergerak (*rolling standard deviation*) antara data asli dan data hasil imputasi, untuk membuktikan bahwa algoritma *volatility-aware* berhasil menjaga struktur varians data.
3. **Perbandingan distribusi kepadatan (KDE)** — membandingkan estimasi kepadatan kernel (*Kernel Density Estimate*) dari distribusi data asli dan data pasca-imputasi, untuk menilai kesesuaian bentuk distribusi secara keseluruhan.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

def visualisasi_pembuktian_imputasi_terpisah(df_original, df_imputed, ticker, window=20):
    # Ambil data 1 saham
    orig = df_original[ticker]
    impt = df_imputed[ticker]
    missing_mask = orig.isna()

    # Cari area zoom (lokasi di mana ada missing value beruntun)
    # Jika tidak ada missing_mask (misal data sudah full), atur fallback ke full index
    try:
        zoom_start = missing_mask[missing_mask].index[0] - pd.Timedelta(days=50)
        zoom_end = missing_mask[missing_mask].index[-1] + pd.Timedelta(days=50)
    except IndexError:
        zoom_start = orig.index[0]
        zoom_end = orig.index[-1]

    plt.style.use('seaborn-v0_8-whitegrid')

    # ==========================================
    # GAMBAR 1: Trajektori Log-Return (Zoom In)
    # ==========================================
    plt.figure(figsize=(10, 5))
    plt.plot(impt[zoom_start:zoom_end].index, impt[zoom_start:zoom_end], color='red', alpha=0.7, label='Data Hasil Imputasi (Multi-Step)')
    plt.plot(orig[zoom_start:zoom_end].index, orig[zoom_start:zoom_end], color='blue', linewidth=1.5, label='Data Observasi Asli')
    # plt.title(f'1. Rekonstruksi Trajektori Log-Return Saham {ticker}\n(Kecocokan Pola Volatilitas Lokal)', fontsize=14, fontweight='bold')
    plt.legend()
    plt.tight_layout()
    plt.show() # Tampilkan gambar pertama

    # ==========================================
    # GAMBAR 2: Preservasi Volatilitas (Rolling Std)
    # ==========================================
    roll_std_orig = orig.rolling(window).std()
    roll_std_impt = impt.rolling(window).std()

    plt.figure(figsize=(10, 5))
    plt.plot(roll_std_impt.index, roll_std_impt, color='red', alpha=0.7, linestyle='--', label='Volatilitas Data Imputasi')
    plt.plot(roll_std_orig.index, roll_std_orig, color='blue', alpha=0.7, label='Volatilitas Data Asli')
    # plt.title(f'2. Preservasi Struktur Varians Saham {ticker}\n(Bukti Keberhasilan Algoritma Volatility-Aware)', fontsize=14, fontweight='bold')
    plt.legend()
    plt.tight_layout()
    plt.show() # Tampilkan gambar kedua

    # ==========================================
    # GAMBAR 3: Distribusi Kepadatan (KDE Plot)
    # ==========================================
    plt.figure(figsize=(10, 6))
    sns.kdeplot(orig.dropna(), color='blue', fill=True, alpha=0.2, label='Distribusi Asli')
    sns.kdeplot(impt, color='red', fill=True, alpha=0.2, label='Distribusi Pasca-Imputasi')
    # plt.title(f'3. Perbandingan Distribusi Kepadatan (KDE) Saham {ticker}', fontsize=14, fontweight='bold')
    plt.legend()
    plt.tight_layout()
    plt.show() # Tampilkan gambar ketiga

# CARA PENGGUNAAN:
visualisasi_pembuktian_imputasi_terpisah(log_return, log_return_imputed, 'AMRT')

## 8. Preprocessing Lanjutan dan Uji Asumsi Statistik

Setelah data log-return lengkap (bebas nilai hilang), tahap berikutnya adalah menguji asumsi-asumsi statistik yang relevan bagi pemodelan deret waktu finansial, yaitu **stasioneritas** dan **efek ARCH (heteroskedastisitas bersyarat)**. Pengujian ini menjadi dasar untuk menentukan kelayakan penggunaan model ARMA–GARCH pada masing-masing saham.

Pustaka `arch` diinstal terlebih dahulu karena dibutuhkan pada tahap pemodelan GARCH selanjutnya.

In [ ]:
log_return

In [ ]:
pip install arch

### 8.1 Tabel Statistik Deskriptif dan Uji Asumsi

Disusun sebuah tabel rekapitulasi (gaya tabel jurnal akademik) yang memuat, untuk setiap saham:

- **Statistik deskriptif**: rata-rata (*mean*), deviasi standar, *skewness*, dan *kurtosis* dari log-return. Nilai kurtosis dihitung dengan basis kurtosis normal = 3 (bukan *excess kurtosis*), sesuai konvensi pelaporan distribusi leptokurtik pada data finansial.
- **Uji *Augmented Dickey-Fuller* (ADF)**: untuk menguji stasioneritas data log-return. Hipotesis nol uji ADF adalah adanya unit root (data tidak stasioner).
- **Uji ARCH-LM (*Lagrange Multiplier*)** dengan lag 5: untuk menguji keberadaan efek heteroskedastisitas bersyarat (ARCH) pada residual, yang menjadi indikasi awal kelayakan pemodelan GARCH.

Tingkat signifikansi statistik ditandai secara otomatis menggunakan notasi bintang gaya jurnal (`***` p<0.01, `**` p<0.05, `*` p<0.1).

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import het_arch
from scipy.stats import skew, kurtosis

def assign_stars(p_val):
    """Fungsi helper untuk otomatisasi bintang signifikansi gaya Jurnal"""
    if p_val < 0.01:
        return '***'
    elif p_val < 0.05:
        return '**'
    elif p_val < 0.1:
        return '*'
    else:
        return ''

def tabel_super_deskriptif_asumsi(df_log_return):
    print(f"Memproses {df_log_return.shape[1]} kolom saham untuk Tabel Super LaTeX...\n")
    rekap_hasil = []

    for kolom in df_log_return.columns:
        log_ret = df_log_return[kolom].dropna()
        residual = log_ret - log_ret.mean()

        # 1. Statistik Deskriptif
        mean_val = log_ret.mean()
        std_val = log_ret.std()
        skew_val = skew(log_ret)
        # fisher=False agar Kurtosis Normal = 3 (sesuai narasi leptokurtik kita)
        kurt_val = kurtosis(log_ret, fisher=False)

        # 2. ADF Test
        adf_result = adfuller(log_ret)
        t_stat_adf = adf_result[0]
        p_val_adf = adf_result[1]

        # 3. ARCH-LM Test (Lag 5)
        arch_result = het_arch(residual, nlags=5)
        lm_stat_arch = arch_result[0]
        p_val_arch = arch_result[1]

        # 4. Formatting Khusus LaTeX (Otomatis pakai bintang)
        adf_latex = f"{t_stat_adf:.4f}{assign_stars(p_val_adf)}"
        arch_pval_latex = f"{p_val_arch:.4f}{assign_stars(p_val_arch)}"

        rekap_hasil.append({
            'Ticker': kolom,
            'Mean': round(mean_val, 4),
            'Std. Deviasi': round(std_val, 4),
            'Skewness': round(skew_val, 3),
            'Kurtosis': round(kurt_val, 3),
            'ADF Stat.': adf_latex,
            'P-Value ARCH-LM': arch_pval_latex
        })

    return pd.DataFrame(rekap_hasil)

# Eksekusi
df_tabel_super = tabel_super_deskriptif_asumsi(log_return_imputed)
df_tabel_super # Tampilkan 5 teratas untuk di-copy ke LaTeX

In [ ]:
df_tabel_super

## 9. Pemilihan Ordo ARMA Terbaik

Untuk setiap saham, dilakukan pencarian ordo ARMA(p, q) terbaik melalui *grid search* terhadap kombinasi parameter autoregresif (p) dan *moving average* (q), dengan kriteria pemilihan berupa nilai **AIC (*Akaike Information Criterion*)** terkecil. Karena data log-return sudah bersifat stasioner, parameter differencing diset *d = 0*, sehingga model yang digunakan setara dengan ARMA(p, q).

Batas pencarian ditetapkan pada $p_{max} = 4$ dan $q_{max} = 4$ sebagai kompromi antara cakupan model dan efisiensi komputasi.

In [ ]:
import pandas as pd
import warnings
from statsmodels.tsa.arima.model import ARIMA

# Kita sembunyikan warning agar output konsol tetap rapi dan tidak spam
warnings.filterwarnings("ignore")

def cari_arma_terbaik_semua_saham(df_log_return, p_max=3, q_max=3):
    """
    Mencari ordo ARMA(p,q) dengan nilai AIC terkecil untuk setiap saham.
    p_max dan q_max membatasi batas pencarian agar komputasi tidak terlalu lama.
    """
    hasil_arma = []

    print(f"Mencari ordo ARMA terbaik (Maksimal p={p_max}, q={q_max}) untuk {len(df_log_return.columns)} saham...\n")
    print("Proses ini mungkin memakan waktu beberapa menit karena mencoba banyak kombinasi model.\n")

    for kolom in df_log_return.columns:
        data = df_log_return[kolom].dropna()

        best_aic = float("inf") # Set awal AIC ke angka tak terhingga
        best_order = None

        # Looping kombinasi p dan q
        for p in range(p_max + 1):
            for q in range(q_max + 1):
                try:
                    # Ordo (p, 0, q) karena data sudah stasioner (d=0)
                    model = ARIMA(data, order=(p, 0, q))
                    res = model.fit()

                    # Cek apakah AIC model ini lebih baik (lebih kecil)
                    if res.aic < best_aic:
                        best_aic = res.aic
                        best_order = (p, q)
                except Exception:
                    # Lewati jika kombinasi ini gagal konvergen secara matematis
                    continue

        # Simpan hasil terbaik untuk saham ini
        hasil_arma.append({
            'Ticker': kolom,
            'Ordo AR (p)': best_order[0] if best_order else None,
            'Ordo MA (q)': best_order[1] if best_order else None,
            'Model Terbaik': f"ARMA({best_order[0]},{best_order[1]})" if best_order else "Gagal",
            'Nilai AIC': round(best_aic, 2) if best_order else None
        })

        # Print progress bar sederhana
        print(f"✅ {kolom} selesai -> Terbaik: ARMA({best_order[0]},{best_order[1]}) | AIC: {best_aic:.2f}")

    # Kembalikan sebagai DataFrame
    df_hasil_arma = pd.DataFrame(hasil_arma)
    return df_hasil_arma

# ==========================================================
# EKSEKUSI
# ==========================================================
# Saya set p_max=2 dan q_max=2 agar cepat.
# Kamu bisa naikkan ke 3, 4, atau 5 jika butuh pencarian lebih luas.
df_ordo_arma = cari_arma_terbaik_semua_saham(log_return_imputed, p_max=4, q_max=4)

print("\n=== RINGKASAN ORDO ARMA TERBAIK ===")
display(df_ordo_arma)

Hasil pencarian ordo ARMA terbaik untuk setiap saham disimpan ke dalam berkas CSV agar dapat digunakan kembali pada tahap pemodelan ARMA–GARCH tanpa perlu mengulang proses pencarian.

In [ ]:
df_ordo_arma.to_csv('df_ordo_arma.csv', index=False)

## 10. Pemodelan ARMA–GARCH dan Estimasi *Value at Risk*

Pustaka `arch` diinstal ulang untuk memastikan ketersediaannya pada lingkungan eksekusi saat ini.

In [ ]:
pip install arch

### 10.1 Pipeline *Sequential*: ARMA (Versi Awal)

Pipeline pertama membangun model ARMA–GARCH secara **dua tahap berurutan (*sequential / two-step*)**:

1. **Persamaan rataan (*mean equation*)** — model ARMA(p, q) diestimasi menggunakan ordo terbaik yang telah ditentukan sebelumnya untuk masing-masing saham, menghasilkan residual dan prediksi rataan satu langkah ke depan.
2. **Persamaan varians (*variance equation*)** — model GARCH diestimasi dari residual ARMA, dengan *grid search* terhadap ordo GARCH(p, q) (hingga orde 2) dan dua pilihan distribusi inovasi: **Normal** dan **Student-t**. Distribusi Student-t disertakan karena lebih mampu menangkap ekor tebal (*heavy tails*) yang umum pada data return finansial.
3. **Uji diagnostik** — residual terstandarisasi dari model GARCH terbaik diuji kembali menggunakan **ARCH-LM** (memastikan tidak ada sisa efek ARCH) dan **Ljung-Box** (memastikan tidak ada autokorelasi residual yang tersisa).
4. **Peramalan VaR** — volatilitas hari berikutnya diramalkan dari model GARCH, kemudian VaR 95% dan VaR 99% dihitung menggunakan rumus parametrik:

$$VaR_{\alpha} = \mu_{t+1} + z_{\alpha} \cdot \sigma_{t+1}$$

dengan $z_{\alpha}$ disesuaikan terhadap distribusi inovasi yang terpilih (kuantil normal atau kuantil Student-t).

In [ ]:
import pandas as pd
import numpy as np
import warnings
from scipy.stats import norm
from statsmodels.tsa.arima.model import ARIMA
from arch import arch_model
from statsmodels.stats.diagnostic import het_arch, acorr_ljungbox

# Sembunyikan warning agar rapi
warnings.filterwarnings("ignore")

def full_pipeline_arma_garch_var(df_log_return, df_ordo_arma, p_garch_max=2, q_garch_max=2):
    hasil_master = []
    df_return_pct = df_log_return

    # Nilai Z-Score awal (akan diupdate otomatis jika pakai distribusi t)
    z_95_norm = norm.ppf(0.05)
    z_99_norm = norm.ppf(0.01)

    print("🚀 Memulai Master Pipeline: ARMA -> GARCH (Grid Search Dist & Ordo) -> VaR...\n")

    for index, baris in df_ordo_arma.iterrows():
        kolom = baris['Ticker']
        p_arma = int(baris['Ordo AR (p)']) if pd.notna(baris['Ordo AR (p)']) else 0
        q_arma = int(baris['Ordo MA (q)']) if pd.notna(baris['Ordo MA (q)']) else 0

        data_saham = df_return_pct[kolom].dropna()

        try:
            # 1. MEAN EQUATION (ARMA)
            model_arma = ARIMA(data_saham, order=(p_arma, 0, q_arma))
            res_arma = model_arma.fit()
            residual_arma = res_arma.resid
            pred_mean = res_arma.forecast(steps=1).iloc[0]

            # 2. VARIANCE EQUATION (Grid Search Ordo & Distribusi)
            best_aic = float("inf")
            best_res_garch = None
            best_p_garch, best_q_garch = 1, 1
            best_dist = 'Normal'

            # Kita coba Normal dan Student-t (t)
            for d in ['Normal', 't']:
                for p_g in range(1, p_garch_max + 1):
                    for q_g in range(1, q_garch_max + 1):
                        try:
                            # dist='t' membantu menangkap outlier/heavy tails yang sering bikin ARCH gagal
                            model_garch = arch_model(residual_arma, mean='Zero', vol='Garch',
                                                     p=p_g, q=q_g, dist=d)
                            res_garch = model_garch.fit(disp='off')

                            if res_garch.aic < best_aic:
                                best_aic = res_garch.aic
                                best_res_garch = res_garch
                                best_p_garch, best_q_garch = p_g, q_g
                                best_dist = d
                        except: continue

            if best_res_garch is None: raise ValueError("GARCH gagal konvergen")

            # 3. UJI DIAGNOSTIK
            std_resid = best_res_garch.resid / best_res_garch.conditional_volatility
            std_resid = std_resid.dropna()

            pval_arch = het_arch(std_resid, nlags=5)[1]
            status_arch = "✅ Lolos" if pval_arch > 0.05 else "❌ Gagal"

            pval_lb = acorr_ljungbox(std_resid, lags=[5])['lb_pvalue'].iloc[0]
            status_lb = "✅ Lolos" if pval_lb > 0.05 else "❌ Gagal"

            # 4. PERAMALAN VaR
            forecast_garch = best_res_garch.forecast(horizon=1)
            pred_volatility = np.sqrt(forecast_garch.variance.iloc[-1, 0])

            # Penyesuaian Z-Score jika menggunakan distribusi Student-t
            if best_dist == 't':
                from scipy.stats import t
                nu = best_res_garch.params['nu'] # Derajat kebebasan (degree of freedom)
                z_95 = t.ppf(0.05, nu)
                z_99 = t.ppf(0.01, nu)
            else:
                z_95, z_99 = z_95_norm, z_99_norm

            var_95 = pred_mean + (z_95 * pred_volatility)
            var_99 = pred_mean + (z_99 * pred_volatility)

            hasil_master.append({
                'Ticker': kolom,
                'Model Final': f"ARMA({p_arma},{q_arma})-GARCH({best_p_garch},{best_q_garch})",
                'Dist': best_dist,
                'AIC': round(best_aic, 2),
                'Diagnostik ARCH': status_arch,
                'Diagnostik Ljung-Box': status_lb,
                'Prediksi Volatilitas (%)': round(pred_volatility, 4),
                'VaR 95% (%)': round(var_95, 4),
                'VaR 99% (%)': round(var_99, 4)
            })
            print(f"✅ {kolom} ({best_dist}) -> ARCH: {status_arch} | VaR 95%: {var_95:.2%}")

        except Exception as e:
            print(f"❌ {kolom} Error: {str(e)}")

    return pd.DataFrame(hasil_master)

# ==========================================================
# EKSEKUSI PIPELINE
# Pastikan variabel `log_return` dan `df_ordo_arma` sudah tersedia
# ==========================================================
df_hasil_akhir = full_pipeline_arma_garch_var(log_return_imputed, df_ordo_arma, p_garch_max=2, q_garch_max=2)

print("\n" + "="*70)
print("🏆 HASIL AKHIR: MODEL TERBAIK, DIAGNOSTIK, DAN PERAMALAN VaR")
print("="*70)
display(df_hasil_akhir)

### 10.2 Penyusunan Ulang Pipeline *Sequential*

Fungsi pipeline *sequential* didefinisikan ulang pada cell berikut untuk memastikan konsistensi definisi fungsi sebelum digunakan pada perbandingan dengan pipeline alternatif di bagian selanjutnya.

In [ ]:
import pandas as pd
import numpy as np
import warnings
from scipy.stats import norm
from statsmodels.tsa.arima.model import ARIMA
from arch import arch_model
from statsmodels.stats.diagnostic import het_arch, acorr_ljungbox

# Sembunyikan warning agar rapi
warnings.filterwarnings("ignore")

def full_pipeline_arma_garch_var(df_log_return, df_ordo_arma, p_garch_max=2, q_garch_max=2):
    hasil_master = []
    df_return_pct = df_log_return

    # Nilai Z-Score awal (akan diupdate otomatis jika pakai distribusi t)
    z_95_norm = norm.ppf(0.05)
    z_99_norm = norm.ppf(0.01)

    print("🚀 Memulai Master Pipeline: ARMA -> GARCH (Grid Search Dist & Ordo) -> VaR...\n")

    for index, baris in df_ordo_arma.iterrows():
        kolom = baris['Ticker']
        p_arma = int(baris['Ordo AR (p)']) if pd.notna(baris['Ordo AR (p)']) else 0
        q_arma = int(baris['Ordo MA (q)']) if pd.notna(baris['Ordo MA (q)']) else 0

        data_saham = df_return_pct[kolom].dropna()

        try:
            # 1. MEAN EQUATION (ARMA)
            model_arma = ARIMA(data_saham, order=(p_arma, 0, q_arma))
            res_arma = model_arma.fit()
            residual_arma = res_arma.resid
            pred_mean = res_arma.forecast(steps=1).iloc[0]

            # 2. VARIANCE EQUATION (Grid Search Ordo & Distribusi)
            best_aic = float("inf")
            best_res_garch = None
            best_p_garch, best_q_garch = 1, 1
            best_dist = 'Normal'

            # Kita coba Normal dan Student-t (t)
            for d in ['Normal', 't']:
                for p_g in range(1, p_garch_max + 1):
                    for q_g in range(1, q_garch_max + 1):
                        try:
                            # dist='t' membantu menangkap outlier/heavy tails yang sering bikin ARCH gagal
                            model_garch = arch_model(residual_arma, mean='Zero', vol='Garch',
                                                     p=p_g, q=q_g, dist=d)
                            res_garch = model_garch.fit(disp='off')

                            if res_garch.aic < best_aic:
                                best_aic = res_garch.aic
                                best_res_garch = res_garch
                                best_p_garch, best_q_garch = p_g, q_g
                                best_dist = d
                        except: continue

            if best_res_garch is None: raise ValueError("GARCH gagal konvergen")

            # 3. UJI DIAGNOSTIK
            std_resid = best_res_garch.resid / best_res_garch.conditional_volatility
            std_resid = std_resid.dropna()

            pval_arch = het_arch(std_resid, nlags=5)[1]
            status_arch = "✅ Lolos" if pval_arch > 0.05 else "❌ Gagal"

            pval_lb = acorr_ljungbox(std_resid, lags=[5])['lb_pvalue'].iloc[0]
            status_lb = "✅ Lolos" if pval_lb > 0.05 else "❌ Gagal"

            # 4. PERAMALAN VaR
            forecast_garch = best_res_garch.forecast(horizon=1)
            pred_volatility = np.sqrt(forecast_garch.variance.iloc[-1, 0])

            # Penyesuaian Z-Score jika menggunakan distribusi Student-t
            if best_dist == 't':
                from scipy.stats import t
                nu = best_res_garch.params['nu'] # Derajat kebebasan (degree of freedom)
                z_95 = t.ppf(0.05, nu)
                z_99 = t.ppf(0.01, nu)
            else:
                z_95, z_99 = z_95_norm, z_99_norm

            var_95 = pred_mean + (z_95 * pred_volatility)
            var_99 = pred_mean + (z_99 * pred_volatility)

            hasil_master.append({
                'Ticker': kolom,
                'Model Final': f"ARMA({p_arma},{q_arma})-GARCH({best_p_garch},{best_q_garch})",
                'Dist': best_dist,
                'AIC': round(best_aic, 2),
                'Diagnostik ARCH': status_arch,
                'Diagnostik Ljung-Box': status_lb,
                'Prediksi Volatilitas (%)': round(pred_volatility, 4),
                'VaR 95% (%)': round(var_95, 4),
                'VaR 99% (%)': round(var_99, 4)
            })
            print(f"✅ {kolom} ({best_dist}) -> ARCH: {status_arch} | VaR 95%: {var_95:.2%}")

        except Exception as e:
            print(f"❌ {kolom} Error: {str(e)}")

    return pd.DataFrame(hasil_master)

### 10.3 Pipeline *Simultaneous* (Joint): AR–GARCH/EGARCH/GJR-GARCH

Sebagai pembanding, dibangun pipeline kedua yang mengestimasi persamaan rataan dan varians secara **simultan (*joint estimation*)** menggunakan satu model `arch_model` dengan mean `'AR'`. Pipeline ini memperluas ruang pencarian model dengan mempertimbangkan beberapa varian model volatilitas:

- **GARCH** standar,
- **EGARCH** (*Exponential GARCH*), yang mampu menangkap asimetri respons volatilitas terhadap *shock* positif dan negatif,
- **GJR-GARCH**, varian GARCH dengan parameter asimetri (`o`), yang juga menangkap efek *leverage* (volatilitas yang meningkat lebih tajam akibat *shock* negatif).

Data discaling (dikalikan 100) sebelum estimasi untuk meningkatkan stabilitas numerik proses optimisasi, kemudian dikembalikan ke skala asli pada tahap peramalan. Pemilihan model terbaik, uji diagnostik (ARCH-LM dan Ljung-Box), serta perhitungan VaR mengikuti logika yang sama seperti pipeline *sequential*, agar hasil keduanya dapat diperbandingkan secara *apple-to-apple*.

Fungsi ini didefinisikan dalam cell yang sama dengan versi terbaru pipeline *sequential*, agar keduanya konsisten secara format keluaran (nama kolom `ARCH-LM` dan `Ljung-Box`) sebelum digabungkan pada tahap pemilihan model hibrida.

In [ ]:
# 1. Update Pipeline Master (Sequential)
def full_pipeline_arma_garch_var(df_log_return, df_ordo_arma, p_garch_max=2, q_garch_max=2):
    hasil_master = []
    print("🚀 Memulai Master Pipeline (Sequential)...")

    for index, baris in df_ordo_arma.iterrows():
        kolom = baris['Ticker']
        p_arma = int(baris['Ordo AR (p)']) if pd.notna(baris['Ordo AR (p)']) else 0
        q_arma = int(baris['Ordo MA (q)']) if pd.notna(baris['Ordo MA (q)']) else 0
        data_saham = df_log_return[kolom].dropna()

        try:
            model_arma = ARIMA(data_saham, order=(p_arma, 0, q_arma))
            res_arma = model_arma.fit()
            residual_arma = res_arma.resid
            pred_mean = res_arma.forecast(steps=1).iloc[0]

            best_aic, best_res_garch, best_dist = float("inf"), None, 'Normal'
            best_p_garch, best_q_garch = 1, 1

            for d in ['Normal', 't']:
                for p_g in range(1, p_garch_max + 1):
                    for q_g in range(1, q_garch_max + 1):
                        try:
                            model_garch = arch_model(residual_arma, mean='Zero', vol='Garch', p=p_g, q=q_g, dist=d)
                            res_garch = model_garch.fit(disp='off', show_warning=False)
                            if res_garch.aic < best_aic:
                                best_aic, best_res_garch = res_garch.aic, res_garch
                                best_p_garch, best_q_garch, best_dist = p_g, q_g, d
                        except: continue

            if best_res_garch is None: continue

            # Diagnostik
            std_resid = (best_res_garch.resid / best_res_garch.conditional_volatility).dropna()
            status_arch = "✅ Lolos" if het_arch(std_resid, nlags=5)[1] > 0.05 else "❌ Gagal"
            status_lb = "✅ Lolos" if acorr_ljungbox(std_resid, lags=[5])['lb_pvalue'].iloc[0] > 0.05 else "❌ Gagal"

            # Peramalan VaR (Simpan dalam Persen agar konsisten)
            forecast_garch = best_res_garch.forecast(horizon=1)
            pred_volatility = np.sqrt(forecast_garch.variance.iloc[-1, 0])

            if best_dist == 't':
                nu = best_res_garch.params['nu']
                z95, z99 = norm.ppf(0.05), norm.ppf(0.01) # fallback norm jika t gagal ppf
                from scipy.stats import t as student_t
                z95, z99 = student_t.ppf(0.05, nu), student_t.ppf(0.01, nu)
            else:
                z95, z99 = norm.ppf(0.05), norm.ppf(0.01)

            # STANDARISASI: Kalikan 100 agar sama dengan Simultaneous
            var_95 = (pred_mean + (z95 * pred_volatility)) * 100
            var_99 = (pred_mean + (z99 * pred_volatility)) * 100

            hasil_master.append({
                'Ticker': kolom,
                'Model': f"ARMA({p_arma},{q_arma})-GARCH({best_p_garch},{best_q_garch})",
                'Dist': best_dist,
                'AIC': best_aic,
                'ARCH-LM': status_arch,
                'Ljung-Box': status_lb,
                'VaR 95% (%)': round(var_95, 4),
                'VaR 99% (%)': round(var_99, 4)
            })
        except Exception as e: print(f"❌ {kolom} Error: {e}")
    return pd.DataFrame(hasil_master)

# 2. Update Pipeline Simultaneous (Joint)
def full_pipeline_simultaneous_ar_garch(df_log_return, df_ordo_arma, p_max=2, q_max=2):
    hasil_master = []
    print("🚀 Memulai Simultaneous Pipeline (Joint)...")

    for index, baris in df_ordo_arma.iterrows():
        kolom = baris['Ticker']
        p_ar_target = int(baris['Ordo AR (p)']) if pd.notna(baris['Ordo AR (p)']) else 1
        data_saham = df_log_return[kolom].dropna()

        # Scaling untuk stabilitas numerik
        scale = 100
        scaled_data = data_saham * scale

        best_aic, best_res, best_config = float("inf"), None, {}

        for d in ['Normal', 't']:
            for v in ['Garch', 'EGarch']:
                for p_g in range(1, p_max + 1):
                    for q_g in range(1, q_max + 1):
                        o_list = [0, 1] if v == 'Garch' else [0]
                        for o_g in o_list:
                            try:
                                model = arch_model(scaled_data, mean='AR', lags=p_ar_target, vol=v, p=p_g, o=o_g, q=q_g, dist=d)
                                res = model.fit(disp='off', show_warning=False)
                                if res.aic < best_aic:
                                    best_aic, best_res = res.aic, res
                                    m_name = "GJR-GARCH" if (v == 'Garch' and o_g > 0) else v
                                    best_config = {'name': m_name, 'p': p_g, 'o': o_g, 'q': q_g, 'dist': d}
                            except: continue
        try:
            if best_res is None: continue
            std_resid = (best_res.resid / best_res.conditional_volatility).dropna()
            status_arch = "✅ Lolos" if het_arch(std_resid, nlags=5)[1] > 0.05 else "❌ Gagal"
            status_lb = "✅ Lolos" if acorr_ljungbox(std_resid, lags=[5])['lb_pvalue'].iloc[0] > 0.05 else "❌ Gagal"

            forecast = best_res.forecast(horizon=1)
            pred_mu = forecast.mean.iloc[-1, 0] / scale
            pred_vol = np.sqrt(forecast.variance.iloc[-1, 0]) / scale

            if best_config['dist'] == 't':
                from scipy.stats import t as student_t
                nu = best_res.params['nu']
                z95, z99 = student_t.ppf(0.05, nu), student_t.ppf(0.01, nu)
            else:
                z95, z99 = norm.ppf(0.05), norm.ppf(0.01)

            var95 = (pred_mu + (z95 * pred_vol)) * 100
            var99 = (pred_mu + (z99 * pred_vol)) * 100 # FIX: gunakan z99, bukan norm.ppf(0.01)

            hasil_master.append({
                'Ticker': kolom,
                'Model': f"AR({p_ar_target})-{best_config['name']}",
                'Dist': best_config['dist'],
                'AIC': best_aic,
                'ARCH-LM': status_arch,
                'Ljung-Box': status_lb,
                'VaR 95% (%)': round(var95, 4),
                'VaR 99% (%)': round(var99, 4)
            })
        except Exception as e: print(f"❌ {kolom} Error: {e}")
    return pd.DataFrame(hasil_master)

## 11. Pemilihan Model Hibrida (*Hybrid Best Model Selection*)

Karena kedua pipeline (*sequential* dan *simultaneous*) memiliki kelebihan masing-masing — *sequential* lebih sederhana dan stabil secara komputasi, sedangkan *simultaneous* lebih fleksibel dalam menangkap asimetri volatilitas — maka dirancang sebuah logika pemilihan model **hibrida** untuk setiap saham, dengan aturan sebagai berikut:

1. Jika hanya satu pipeline yang lolos kedua uji diagnostik (ARCH-LM dan Ljung-Box), pipeline tersebut yang dipilih.
2. Jika kedua pipeline lolos, dipilih model dengan nilai **AIC terkecil** (model yang relatif lebih *fit* terhadap data).
3. Jika kedua pipeline gagal lolos uji diagnostik, pipeline *sequential* (Master) digunakan sebagai solusi cadangan (*fallback best-effort*), dengan catatan keterbatasan ini perlu disebutkan pada interpretasi hasil.

Pendekatan ini memastikan bahwa model akhir yang digunakan untuk estimasi VaR setiap saham adalah model dengan kualitas statistik terbaik yang tersedia di antara dua alternatif yang diuji.

In [ ]:
def hybrid_best_var_pipeline(df_log_return, df_ordo_arma):
    """
    Fungsi Hybrid yang sudah disinkronkan dengan nama kolom baru:
    'ARCH-LM' dan 'Ljung-Box' untuk kedua pipeline.
    """
    # 1. Jalankan Master Pipeline (Sequential)
    df_master = full_pipeline_arma_garch_var(df_log_return, df_ordo_arma)

    # 2. Jalankan Simultaneous Pipeline (Joint Advanced)
    df_simul = full_pipeline_simultaneous_ar_garch(df_log_return, df_ordo_arma)

    hasil_final = []
    tickers = df_log_return.columns

    for ticker in tickers:
        # Ambil data dari masing-masing pipeline
        row_m = df_master[df_master['Ticker'] == ticker].iloc[0] if ticker in df_master['Ticker'].values else None
        row_s = df_simul[df_simul['Ticker'] == ticker].iloc[0] if ticker in df_simul['Ticker'].values else None

        # --- LOGIKA EVALUASI KELOLOSAN ---
        # Sekarang keduanya menggunakan nama kolom yang sama: 'ARCH-LM' & 'Ljung-Box'
        m_pass = (row_m['ARCH-LM'] == "✅ Lolos" and row_m['Ljung-Box'] == "✅ Lolos") if row_m is not None else False
        s_pass = (row_s['ARCH-LM'] == "✅ Lolos" and row_s['Ljung-Box'] == "✅ Lolos") if row_s is not None else False

        # --- LOGIKA PEMILIHAN MODEL TERBAIK ---
        if s_pass and not m_pass:
            chosen, source = row_s, "Simultaneous (Advanced)"
        elif m_pass and not s_pass:
            chosen, source = row_m, "Master (ARMA-GARCH)"
        elif m_pass and s_pass:
            # Jika keduanya lolos, pilih yang AIC-nya paling rendah (lebih fit)
            if row_s['AIC'] < row_m['AIC']:
                chosen, source = row_s, "Simultaneous (Lower AIC)"
            else:
                chosen, source = row_m, "Master (Lower AIC)"
        else:
            # Jika keduanya gagal, ambil Master sebagai fallback (best effort)
            chosen = row_m if row_m is not None else row_s
            source = "⚠️ Both Failed (Using Fallback)"

        if chosen is not None:
            hasil_final.append({
                'Ticker': ticker,
                'Method': source,
                'Model Final': chosen['Model'],
                'Dist': chosen['Dist'],
                'AIC': round(chosen['AIC'], 2),
                'ARCH': chosen['ARCH-LM'],
                'LB': chosen['Ljung-Box'],
                'VaR 95% (%)': chosen['VaR 95% (%)'],
                'VaR 99% (%)': chosen['VaR 99% (%)']
            })

    return pd.DataFrame(hasil_final)

# ==========================================================
# EKSEKUSI ULANG
# ==========================================================
# Pastikan log_return_imputed dan df_ordo_arma sudah terdefinisi
df_hybrid_final = hybrid_best_var_pipeline(log_return_imputed, df_ordo_arma)

print("\n" + "="*80)
print("🏆 HASIL FINAL: MODEL TERBAIK UNTUK TIAP SAHAM")
print("="*80)
display(df_hybrid_final)

Hasil model hibrida final untuk seluruh saham disimpan ke dalam berkas Excel sebagai keluaran antara (*intermediate output*) yang dapat ditelusuri ulang.

In [ ]:
df_hybrid_final.to_excel('hasil_final.xlsx', index=False)

## 12. Simulasi Estimasi Kerugian dalam Nominal Rupiah

Nilai VaR yang dihasilkan pada tahap sebelumnya berbentuk persentase return. Untuk memberikan interpretasi yang lebih intuitif dan relevan secara praktis, nilai VaR persentase dikonversi menjadi estimasi kerugian maksimal dalam nominal Rupiah, dengan asumsi modal investasi tertentu per saham (dalam contoh ini, Rp 10.000.000 per saham).

In [ ]:
import pandas as pd
import numpy as np

def simulasi_var_rupiah(df_hasil_akhir, modal_per_saham=10000000):
    """
    Fungsi untuk mengonversi persentase VaR menjadi estimasi
    maksimal kerugian dalam bentuk nominal Rupiah.
    """
    print(f"💰 Simulasi Risiko untuk Modal: Rp {modal_per_saham:,.0f} per saham\n")

    # Filter hanya saham yang modelnya 'Berhasil' (Drop MAPI atau yang Error)
    df_valid = df_hasil_akhir[df_hasil_akhir['Model Final'] != 'Error'].copy()

    # Hitung nominal kerugian (VaR % dikali Modal)
    # Ingat: VaR % di tabel sebelumnya masih dalam bentuk persentase utuh (misal -2.15),
    # jadi kita bagi 100 dulu untuk perhitungan nominal.
    df_valid['Potensi Rugi 95% (Rp)'] = (df_valid['VaR 95% (%)'] / 100) * modal_per_saham
    df_valid['Potensi Rugi 99% (Rp)'] = (df_valid['VaR 99% (%)'] / 100) * modal_per_saham

    # Tampilkan kolom yang relevan saja agar rapi
    kolom_tampil = ['Ticker', 'VaR 95% (%)', 'Potensi Rugi 95% (Rp)', 'VaR 99% (%)', 'Potensi Rugi 99% (Rp)']
    df_simulasi = df_valid[kolom_tampil]

    return df_simulasi

# ==========================================================
# EKSEKUSI
# Masukkan DataFrame output dari full pipeline sebelumnya
# ==========================================================
modal_investasi = 10000000 # Rp 10 Juta
df_rupiah = simulasi_var_rupiah(df_hybrid_final, modal_investasi)

print("=== ESTIMASI KERUGIAN MAKSIMAL 1-HARI KE DEPAN ===")
display(df_rupiah)

Hasil simulasi disimpan ke dalam berkas Excel, kemudian dibaca kembali sebagai bentuk validasi bahwa proses penyimpanan dan pembacaan data konsisten.

In [ ]:
df_rupiah.to_excel("VaR.xlsx", index=False)

In [ ]:
df_rupiah = pd.read_excel("VaR.xlsx")
df_rupiah

## 13. Analisis Risiko Tingkat Portofolio

Estimasi VaR pada bagian sebelumnya bersifat **individual** (per saham), yang mengasumsikan setiap saham berdiri sendiri tanpa memperhitungkan interaksi (korelasi) antar-saham. Pada bagian ini, dilakukan analisis risiko pada level **portofolio**, dengan memperhitungkan efek diversifikasi melalui dua metode: pendekatan parametrik **Varians-Kovarians** dan pendekatan simulasi **Monte Carlo**.

### 13.1 Matriks Korelasi Antar-Saham

Sebagai langkah awal, dibangun **heatmap matriks korelasi** antar log-return seluruh saham yang valid (model konvergen) dalam portofolio. Visualisasi ini membantu mengidentifikasi pasangan saham dengan korelasi tinggi (risiko terkonsentrasi) maupun rendah/negatif (potensi diversifikasi).

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

def visualisasi_heatmap_korelasi(df_log_return, df_hasil_akhir):
    """
    Membuat visualisasi Heatmap untuk Matriks Korelasi dari saham-saham
    yang valid di dalam portofolio.
    """
    # 1. Filter hanya saham yang Valid (Model Final tidak Error)
    df_valid = df_hasil_akhir[df_hasil_akhir['Model Final'] != 'Error'].copy()
    saham_valid = df_valid['Ticker'].tolist()

    # 2. Hitung Matriks Korelasi
    matriks_korelasi = df_log_return[saham_valid].corr()

    # 3. Setup Figure (Ukuran kanvas besar agar 28 saham terbaca jelas)
    plt.figure(figsize=(14, 12))

    # 4. Buat Masking untuk Segitiga Atas (Opsional tapi direkomendasikan)
    # Ini agar heatmap bentuknya segitiga ke bawah, lebih elegan dan tidak redundan
    mask = np.triu(np.ones_like(matriks_korelasi, dtype=bool))

    # 5. Plot Heatmap menggunakan Seaborn
    # cmap='coolwarm' -> Merah untuk korelasi tinggi, Biru untuk korelasi rendah/negatif
    sns.heatmap(matriks_korelasi,
                mask=mask,
                cmap='coolwarm',
                vmin=-1, vmax=1,
                center=0,
                square=True,
                linewidths=.5,
                cbar_kws={"shrink": .8},
                annot=False) # annot=False karena 28x28 akan terlalu sumpek jika diisi angka

    # 6. Kustomisasi Tampilan
    plt.title('Heatmap Matriks Korelasi Portofolio (Efek Diversifikasi)', fontsize=16, fontweight='bold', pad=20)
    plt.xticks(rotation=45, ha='right', fontsize=9)
    plt.yticks(rotation=0, fontsize=9)

    # Tampilkan plot
    plt.tight_layout()
    plt.show()

# ==========================================================
# EKSEKUSI
# ==========================================================
visualisasi_heatmap_korelasi(log_return_imputed, df_hybrid_final)

### 13.2 VaR Portofolio: Pendekatan Varians-Kovarians (Parametrik)

Pendekatan **Varians-Kovarians** mengestimasi risiko portofolio secara analitik berdasarkan matriks kovarians antar-saham. Karena volatilitas individual ($\sigma_i$) tidak secara langsung tersedia pada tabel hasil model hibrida, nilai tersebut diestimasi kembali (*back-calculated*) dari VaR 95% setiap saham menggunakan hubungan:

$$\sigma_i = \frac{VaR_{95\%,i}}{z_{0.05}}$$

Selanjutnya, matriks kovarians portofolio dibentuk dari matriks korelasi historis dan estimasi volatilitas tersebut, dengan bobot portofolio diasumsikan **sama rata (*equal weight*)** antar-saham. VaR portofolio kemudian dihitung secara parametrik:

$$VaR_{\alpha,\,portofolio} = \mu_{portofolio} + z_{\alpha} \cdot \sigma_{portofolio}$$

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm, t as student_t

def hitung_var_kovarians_portofolio(df_log_return, df_hasil_akhir, modal_total=280000000):
    """
    Menghitung VaR Portofolio Parametrik dengan pemulihan data otomatis (Back-calculate Volatility).
    """
    # 1. Identifikasi Nama Kolom (Adaptif terhadap Tabel Akademik atau Mentah)
    col_ticker = 'Ticker' if 'Ticker' in df_hasil_akhir.columns else 'Emiten'
    col_var95 = 'VaR 95% (%)'
    col_dist = 'Dist' if 'Dist' in df_hasil_akhir.columns else 'Distribusi Inovasi'

    # Filter saham valid
    df_valid = df_hasil_akhir.copy()
    saham_valid = df_valid[col_ticker].tolist()
    jumlah_saham = len(saham_valid)

    # 2. Back-calculate Volatilitas (Sigma) dan Mean (Mu)
    vol_pred = []
    mu_pred = df_log_return[saham_valid].mean().values # Mean harian historis

    for i, row in df_valid.iterrows():
        dist_type = str(row[col_dist]).lower()
        var_val = row[col_var95] / 100 # Ubah ke desimal

        # Ambil Z-score 95% sesuai distribusi
        if 't' in dist_type or 'student' in dist_type:
            z_95 = student_t.ppf(0.05, df=5) # Asumsi df=5 jika tidak tersimpan
        else:
            z_95 = norm.ppf(0.05)

        # Sigma = VaR / Z (Asumsi mu ~ 0 untuk skala harian)
        vol_pred.append(abs(var_val / z_95))

    vol_pred = np.array(vol_pred)

    # 3. Hitung Matriks Korelasi Historis
    korelasi_matrix = df_log_return[saham_valid].corr().values

    # 4. Bentuk Matriks Varian-Kovarians (W_transpose * Cov * W)
    # Cov(i,j) = Corr(i,j) * sigma_i * sigma_j
    cov_matrix = np.outer(vol_pred, vol_pred) * korelasi_matrix

    # 5. Tentukan Bobot (Equal Weight)
    bobot = np.full(jumlah_saham, 1 / jumlah_saham)

    # 6. Hitung Statistik Portofolio
    port_mean = np.dot(bobot, mu_pred)
    port_variance = np.dot(bobot.T, np.dot(cov_matrix, bobot))
    port_volatility = np.sqrt(port_variance)

    # 7. Hitung VaR Parametrik Portofolio
    z_95_p = norm.ppf(0.05)
    z_99_p = norm.ppf(0.01)

    var_95_port = (port_mean + (z_95_p * port_volatility)) * 100
    var_99_port = (port_mean + (z_99_p * port_volatility)) * 100

    # Nominal Rupiah
    rugi_95_rp = (var_95_port / 100) * modal_total
    rugi_99_rp = (var_99_port / 100) * modal_total

    # -- OUTPUT FORMAL --
    print("="*60)
    print("📊 ANALISIS PARAMETRIK: VARIAN-KOVARIANS PORTOFOLIO")
    print("="*60)
    print(f"Modal Total          : Rp {modal_total:,.0f}")
    print(f"Prediksi Return (μ)  : {port_mean * 100:.4f}%")
    print(f"Volatilitas Port (σ) : {port_volatility * 100:.4f}%")
    print("-" * 60)
    print(f"VaR 95% Portofolio   : {var_95_port:.4f}%")
    print(f"Estimasi Maks Rugi   : Rp {abs(rugi_95_rp):,.0f}")
    print(f"VaR 99% Portofolio   : {var_99_port:.4f}%")
    print(f"Estimasi Maks Rugi   : Rp {abs(rugi_99_rp):,.0f}")
    print("="*60)

    # DataFrame untuk Matriks Kovarians
    df_cov = pd.DataFrame(cov_matrix * 10000, index=saham_valid, columns=saham_valid)

    return var_95_port,  var_99_port

# ==========================================================
# EKSEKUSI
# ==========================================================
hitung_var_kovarians_portofolio(log_return_imputed, df_hybrid_final)

### 13.3 VaR Portofolio: Simulasi Monte Carlo

Sebagai pembanding terhadap pendekatan parametrik, dilakukan **simulasi Monte Carlo** dengan membangkitkan ribuan skenario return portofolio (dalam contoh ini, 10.000 skenario) menggunakan distribusi normal multivariat (*multivariate normal*), berdasarkan estimasi rata-rata, volatilitas, dan matriks korelasi historis yang sama seperti pada pendekatan Varians-Kovarians.

VaR portofolio kemudian diestimasi secara empiris dari distribusi hasil simulasi, yaitu persentil ke-5 (untuk VaR 95%) dan persentil ke-1 (untuk VaR 99%) dari sebaran return portofolio simulasi. Pendekatan ini tidak terikat pada asumsi distribusi tertutup secara analitik dan memberikan gambaran visual yang lebih kaya melalui histogram distribusi return portofolio.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, t as student_t

def simulasi_monte_carlo_portofolio(df_log_return, df_hasil_akhir, modal_total=280000000, num_simulations=10000):
    """
    Versi Perbaikan: Menangani KeyError dan sinkronisasi kolom akademik.
    """
    print(f"🎲 Memulai Simulasi Monte Carlo ({num_simulations:,} Skenario)...")

    # 1. Sinkronisasi Nama Kolom (Menangani versi akademik atau versi mentah)
    # Kita cari kolom Ticker/Emiten dan VaR
    col_ticker = 'Ticker' if 'Ticker' in df_hasil_akhir.columns else 'Emiten'
    col_var95 = 'VaR 95% (%)'
    col_dist = 'Dist' if 'Dist' in df_hasil_akhir.columns else 'Distribusi Inovasi'

    # Filter data valid
    df_valid = df_hasil_akhir.copy()
    saham_valid = df_valid[col_ticker].tolist()
    jumlah_saham = len(saham_valid)

    # 2. Estimasi Volatilitas & Mean (Back-calculation)
    # Karena volatilitas tidak ada di tabel, kita hitung dari VaR 95%
    # Rumus: VaR = mu + (z * sigma) -> sigma = VaR / z (asumsi mu harian ~ 0)
    vol_pred = []
    mu_pred = df_log_return[saham_valid].mean().values # Gunakan mean historis agar lebih akurat

    for i, row in df_valid.iterrows():
        dist_type = str(row[col_dist]).lower()
        var_value = row[col_var95] / 100 # ubah ke desimal

        # Tentukan Z-score berdasarkan distribusi
        if 't' in dist_type or 'student' in dist_type:
            # Jika distribusi t, gunakan ppf t (asumsi df/nu default 5 jika tidak ada)
            z_95 = student_t.ppf(0.05, df=5)
        else:
            z_95 = norm.ppf(0.05)

        # Estimasi Sigma (Volatilitas)
        sigma_est = abs(var_value / z_95)
        vol_pred.append(sigma_est)

    vol_pred = np.array(vol_pred)

    # 3. Hitung Matriks Korelasi Historis
    korelasi_historis = df_log_return[saham_valid].corr().values

    # 4. Bangun Matriks Kovarians
    cov_matrix = np.zeros((jumlah_saham, jumlah_saham))
    for i in range(jumlah_saham):
        for j in range(jumlah_saham):
            cov_matrix[i, j] = korelasi_historis[i, j] * vol_pred[i] * vol_pred[j]

    # 5. Simulasi Multivariate Normal
    simulated_returns = np.random.multivariate_normal(mu_pred, cov_matrix, num_simulations)

    # 6. Hitung Return Portofolio (Equal Weight)
    bobot = np.full(jumlah_saham, 1 / jumlah_saham)
    portofolio_returns = np.dot(simulated_returns, bobot) * 100 # Kembalikan ke persen

    # 7. Ekstrak VaR Portofolio
    var_95_port = np.percentile(portofolio_returns, 5)
    var_99_port = np.percentile(portofolio_returns, 1)

    # Nominal Rupiah
    rugi_95_rp = (var_95_port / 100) * modal_total
    rugi_99_rp = (var_99_port / 100) * modal_total

    # Tampilkan Hasil
    print("\n" + "="*55)
    print("🏆 HASIL SIMULASI MONTE CARLO VaR PORTOFOLIO")
    print(f"Modal Investasi : Rp {modal_total:,.0f}")
    print(f"Jumlah Saham    : {jumlah_saham} Emiten (Equal Weight)")
    print("-" * 55)
    print(f"VaR 95% Portofolio : {var_95_port:.4f}%")
    print(f"Potensi Rugi Maks  : Rp {abs(rugi_95_rp):,.0f}")
    print(f"VaR 99% Portofolio : {var_99_port:.4f}%")
    print(f"Potensi Rugi Maks  : Rp {abs(rugi_99_rp):,.0f}")
    print("="*55)

    # 8. Visualisasi Akademik
    plt.figure(figsize=(11, 6))
    n, bins, patches = plt.hist(portofolio_returns, bins=60, color='#2c3e50', alpha=0.7, edgecolor='white')

    # Tambahkan Garis VaR
    plt.axvline(var_95_port, color='#e74c3c', linestyle='--', linewidth=2, label=f'VaR 95% ({var_95_port:.2f}%)')
    plt.axvline(var_99_port, color='#c0392b', linestyle='-', linewidth=2, label=f'VaR 99% ({var_99_port:.2f}%)')

    plt.title(f"Distribusi Probabilitas Return Portofolio\n(Simulasi Monte Carlo: {num_simulations:,} Iterasi)",
              fontsize=14, fontweight='bold')
    plt.xlabel('Prediksi Return Portofolio (%)', fontsize=12)
    plt.ylabel('Frekuensi Skenario', fontsize=12)
    plt.legend(loc='upper right')
    plt.grid(axis='y', alpha=0.2)

    # Area Risiko
    plt.axvspan(portofolio_returns.min(), var_95_port, color='red', alpha=0.1)

    plt.tight_layout()
    plt.show()

    # Tambahkan di baris paling bawah fungsi simulasi_monte_carlo_portofolio
    return var_95_port, var_99_port

# Eksekusi
simulasi_monte_carlo_portofolio(log_return_imputed, df_hybrid_final)

## 14. Komparasi Risiko Individual vs. Portofolio

Sebagai rangkuman akhir, disusun sebuah tabel komparasi (**Tabel 5**) yang membandingkan:

- Rata-rata VaR individual (rata-rata risiko seluruh saham jika dipegang sendiri-sendiri),
- VaR portofolio hasil pendekatan Varians-Kovarians,
- VaR portofolio hasil simulasi Monte Carlo,
- **Persentase pengurangan risiko (efek diversifikasi)** yang diperoleh dari penggabungan saham ke dalam satu portofolio,
- Estimasi kerugian maksimal dalam nominal Rupiah, baik secara individual maupun portofolio (dinormalisasi per porsi saham agar dapat diperbandingkan secara *apple-to-apple*),
- Estimasi penghematan risiko (Rupiah) per saham akibat diversifikasi.

Tabel ini menjadi dasar kesimpulan akhir mengenai manfaat diversifikasi pada portofolio yang dianalisis.

In [ ]:
import pandas as pd
import numpy as np

def buat_tabel_komparasi_risiko(df_hasil_akhir, var_95_vc, var_99_vc, var_95_mc, var_99_mc, modal_total=280000000):
    """
    Script Master untuk merangkum hasil VaR Individual vs Portofolio
    persis seperti format Tabel 5.
    """
    # 1. Ambil jumlah saham dan hitung modal per saham
    jumlah_saham = len(df_hasil_akhir)
    modal_per_saham = modal_total / jumlah_saham

    # 2. Hitung Rata-rata VaR Individual dari tabel sebelumnya
    # Asumsi nama kolom di df_hasil_akhir adalah 'VaR 95% (%)' dan 'VaR 99% (%)'
    rata_var_95_ind = df_hasil_akhir['VaR 95% (%)'].mean()
    rata_var_99_ind = df_hasil_akhir['VaR 99% (%)'].mean()

    # 3. Hitung Efek Diversifikasi (Persentase Pengurangan Risiko)
    # Menggunakan acuan Varians-Kovarians vs Individual
    div_95 = ((abs(var_95_vc) - abs(rata_var_95_ind)) / abs(rata_var_95_ind)) * 100
    div_99 = ((abs(var_99_vc) - abs(rata_var_99_ind)) / abs(rata_var_99_ind)) * 100

    # 4. Kalkulasi Nominal Rupiah (Kerugian Maksimal per Porsi Saham)
    # Rata-rata rugi jika saham berdiri sendiri
    rugi_ind_95_rp = (rata_var_95_ind / 100) * modal_per_saham
    rugi_ind_99_rp = (rata_var_99_ind / 100) * modal_per_saham

    # Rugi portofolio (dibagi per porsi saham agar sepadan/apple-to-apple)
    rugi_port_95_rp = (var_95_vc / 100) * modal_per_saham
    rugi_port_99_rp = (var_99_vc / 100) * modal_per_saham

    # Penghematan Risiko per saham (Selisih)
    hemat_95_rp = abs(rugi_ind_95_rp) - abs(rugi_port_95_rp)
    hemat_99_rp = abs(rugi_ind_99_rp) - abs(rugi_port_99_rp)

    # 5. Susun ke dalam DataFrame
    tabel_5 = {
        'Metrik': [
            'Rata-rata VaR Individual',
            'VaR Portofolio (Varians-Kovarians)',
            'VaR Portofolio (Monte Carlo)',
            'Pengurangan Risiko (Diversifikasi)',
            'Kerugian Max Individual Rata-rata (Rp)',
            'Kerugian Max Portofolio—per saham (Rp)',
            'Penghematan Risiko per Saham (Rp)'
        ],
        'VaR 95%': [
            f"{rata_var_95_ind:.2f}%",
            f"{var_95_vc:.2f}%",
            f"{var_95_mc:.2f}%",
            f"{div_95:.1f}%",
            f"{rugi_ind_95_rp:,.0f}".replace(',', '.'),
            f"{rugi_port_95_rp:,.0f}".replace(',', '.'),
            f"+{hemat_95_rp:,.0f}".replace(',', '.')
        ],
        'VaR 99%': [
            f"{rata_var_99_ind:.2f}%",
            f"{var_99_vc:.2f}%",
            f"{var_99_mc:.2f}%",
            f"{div_99:.1f}%",
            f"{rugi_ind_99_rp:,.0f}".replace(',', '.'),
            f"{rugi_port_99_rp:,.0f}".replace(',', '.'),
            f"+{hemat_99_rp:,.0f}".replace(',', '.')
        ]
    }

    df_tabel_5 = pd.DataFrame(tabel_5)

    return df_tabel_5

# ==========================================================
# CARA EKSEKUSI:
# Pastikan fungsi Monte Carlo dan Var-Cov sudah di-run dan me-return nilai
# ==========================================================

# 1. Jalankan Varians-Kovarians
var_95_vc, var_99_vc = hitung_var_kovarians_portofolio(log_return_imputed, df_hybrid_final)

# 2. Jalankan Monte Carlo
var_95_mc, var_99_mc = simulasi_monte_carlo_portofolio(log_return_imputed, df_hybrid_final)

# 3. Buat Tabel 5
df_komparasi = buat_tabel_komparasi_risiko(
    df_hasil_akhir=df_hybrid_final, # Sesuaikan dengan nama variabel dataframe-mu
    var_95_vc=var_95_vc,
    var_99_vc=var_99_vc,
    var_95_mc=var_95_mc,
    var_99_mc=var_99_mc
)

print("\n" + "="*70)
print("Tabel 5. Perbandingan VaR Individual vs. VaR Portofolio")
print("="*70)
display(df_komparasi)

## 15. Lampiran: Visualisasi Persamaan Rekonstruksi Harga

Sebagai pelengkap dokumentasi, berikut divisualisasikan persamaan yang digunakan untuk merekonstruksi harga dari log-return (lihat Bagian 6, fungsi `reconstruct_price`):

$$P_t = P_{t-1} \times e^{r_t}$$

Visualisasi ini disimpan sebagai berkas gambar (`latex.png`) untuk keperluan penyisipan pada laporan atau dokumen ilmiah dalam format LaTeX.

In [ ]:
import matplotlib.pyplot as plt

text = r"$P_t = P_{t-1} \times e^{r_t}$"

plt.figure(figsize=(4,2))
plt.text(0.5, 0.5, text, fontsize=20, ha='center', va='center')
plt.axis('off')

plt.savefig("latex.png", dpi=300, bbox_inches='tight')
plt.show()

---

## Penutup

Notebook ini mendokumentasikan keseluruhan alur kerja analisis VaR portofolio saham, mulai dari pembersihan data, imputasi nilai hilang yang menjaga karakteristik statistik asli, pengujian asumsi, pemodelan ARMA–GARCH dengan dua pendekatan estimasi (*sequential* dan *simultaneous*), hingga estimasi risiko pada level individual dan portofolio. Hasil akhir menunjukkan estimasi VaR 95% dan VaR 99% untuk setiap saham, beserta estimasi efek diversifikasi pada level portofolio yang dikuantifikasi melalui dua metode independen (Varians-Kovarians dan Monte Carlo) sebagai bentuk validasi silang (*cross-validation*) hasil.

**Tim STAVANTS — Data Analytics Competition (DAC)**
